# 수묵화 데이터셋 전처리 파이프라인

AI Hub 수묵화 데이터셋을 로딩, 검증, 전처리하는 노트북입니다.

## 파이프라인 순서
1. 데이터 로딩 & 기본 탐색
2. 수묵화 특징 검증 필터
3. 전처리 적용 (리사이즈, 정규화, 필터링)
4. ControlNet용 스케치 추출
5. 메타데이터 & 통계 리포트

In [ ]:
# ============================
# 라이브러리 임포트
# ============================
import os
import glob
import json
from pathlib import Path

import numpy as np          # 수치 연산 (배열, 통계 등)
import cv2                  # OpenCV: 이미지 처리 (리사이즈, 엣지검출, 색공간 변환)
from PIL import Image       # 이미지 열기/메타데이터 확인용
import matplotlib.pyplot as plt        # 시각화 (그래프, 이미지 출력)
import matplotlib.font_manager as fm   # 한글 폰트 설정용
from tqdm.notebook import tqdm         # 진행률 표시 바
import pandas as pd         # 데이터프레임으로 이미지 정보/특징값 관리

In [ ]:
# ============================
# 경로 설정
# ============================
# 프로젝트 루트 (notebooks/ 기준 한 단계 위)
PROJECT_ROOT = Path("../")

# data/raw/     → AI Hub에서 다운받은 원본 수묵화 이미지를 여기에 넣으세요
# data/processed/ → 전처리(리사이즈+필터링) 완료된 이미지가 저장됨
# data/sketches/  → ControlNet 학습용 스케치(엣지) 이미지가 저장됨
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SKETCH_DIR = PROJECT_ROOT / "data" / "sketches"

# 폴더가 없으면 자동 생성
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SKETCH_DIR.mkdir(parents=True, exist_ok=True)

# 이미지로 인식할 확장자 목록
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

print(f"RAW_DIR: {RAW_DIR.resolve()}")
print(f"PROCESSED_DIR: {PROCESSED_DIR.resolve()}")
print(f"SKETCH_DIR: {SKETCH_DIR.resolve()}")

## 1. 데이터 로딩 & 기본 탐색

In [ ]:
def collect_image_paths(root_dir: Path) -> list[Path]:
    """
    root_dir 하위의 모든 이미지 파일 경로를 수집한다.
    - 하위 폴더까지 재귀적으로 탐색 (rglob)
    - 대소문자 모두 처리 (.jpg, .JPG 등)
    - 중복 제거 후 정렬하여 반환
    """
    paths = []
    for ext in IMG_EXTENSIONS:
        paths.extend(root_dir.rglob(f"*{ext}"))       # 소문자 확장자
        paths.extend(root_dir.rglob(f"*{ext.upper()}"))  # 대문자 확장자
    return sorted(set(paths))  # 중복 제거 + 정렬


# 원본 이미지 경로 수집
image_paths = collect_image_paths(RAW_DIR)
print(f"총 이미지 수: {len(image_paths)}")

# 이미지가 없으면 안내 메시지 출력
if len(image_paths) == 0:
    print("\n[!] data/raw/ 폴더에 이미지가 없습니다.")
    print("    AI Hub에서 다운로드한 수묵화 데이터셋을 data/raw/ 에 넣어주세요.")

In [ ]:
def get_image_info(path: Path) -> dict:
    """
    이미지 파일 하나의 기본 정보를 딕셔너리로 반환한다.
    - width, height: 이미지 가로/세로 크기 (픽셀)
    - aspect_ratio: 종횡비 (가로/세로)
    - format: 파일 포맷 (JPEG, PNG 등)
    - mode: 색상 모드 (RGB, L 등)
    - filesize_kb: 파일 크기 (KB)
    열 수 없는 이미지는 error 키에 에러 메시지를 담아 반환
    """
    try:
        img = Image.open(path)
        w, h = img.size
        return {
            "path": str(path),
            "filename": path.name,
            "width": w,
            "height": h,
            "aspect_ratio": round(w / h, 2),
            "format": img.format,
            "mode": img.mode,
            "filesize_kb": round(path.stat().st_size / 1024, 1),
        }
    except Exception as e:
        return {"path": str(path), "error": str(e)}


# 전체 이미지의 기본 정보를 수집하여 DataFrame으로 정리
if image_paths:
    infos = [get_image_info(p) for p in tqdm(image_paths, desc="이미지 정보 수집")]
    df_info = pd.DataFrame(infos)

    # 정상 이미지와 오류(손상/열 수 없는) 이미지를 분리
    df_errors = df_info[df_info.get("error", pd.Series(dtype=str)).notna()].copy()
    df_valid = df_info[~df_info.index.isin(df_errors.index)].copy()

    print(f"\n정상 이미지: {len(df_valid)}")
    print(f"오류 이미지: {len(df_errors)}")
    print(f"\n--- 해상도 통계 (min, mean, max 등) ---")
    print(df_valid[["width", "height", "filesize_kb"]].describe())
else:
    print("이미지가 없으므로 건너뜁니다.")

In [ ]:
def show_sample_grid(paths: list[Path], n=16, figsize=(16, 16)):
    """
    랜덤으로 n장을 뽑아 그리드 형태로 출력한다.
    데이터가 어떤 이미지들로 구성되어 있는지 눈으로 확인하는 용도.
    """
    n = min(n, len(paths))
    indices = np.random.choice(len(paths), n, replace=False)  # 랜덤 인덱스
    cols = int(np.ceil(np.sqrt(n)))   # 그리드 열 수
    rows = int(np.ceil(n / cols))     # 그리드 행 수

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        img = Image.open(paths[idx])
        axes[i].imshow(img)
        axes[i].set_title(paths[idx].name, fontsize=8)
        axes[i].axis("off")

    # 남는 칸은 비워두기
    for i in range(n, len(axes)):
        axes[i].axis("off")

    plt.suptitle("랜덤 샘플 이미지", fontsize=14)
    plt.tight_layout()
    plt.show()


if image_paths:
    show_sample_grid(image_paths)

## 2. 수묵화 특징 검증 필터

수묵화의 핵심 특징을 수치화하여 검증합니다:
- **채도(Saturation)**: 수묵화는 무채색 위주 → 낮은 채도
- **여백 비율**: 밝은 배경 영역이 넓음
- **그레이스케일 유사도**: R, G, B 채널 간 차이가 적음
- **엣지 밀도**: 부드러운 붓터치 → 낮은 엣지 밀도

In [ ]:
def compute_ink_features(img_path: Path) -> dict:
    """
    수묵화 판별을 위한 5가지 특징값을 계산한다.

    [수묵화의 시각적 특징]
    - 먹과 물로만 그리므로 채도가 낮다 (거의 무채색)
    - 여백의 미: 빈 공간(흰 배경)이 많다
    - RGB 채널 값이 거의 동일하다 (그레이스케일에 가까움)
    - 붓터치가 부드러워 엣지(경계선)가 적다

    Returns:
        dict:
        - mean_saturation: HSV 채도 평균 (0~255, 낮을수록 무채색)
        - whitespace_ratio: 밝은 픽셀(V>200) 비율 (0~1, 높을수록 여백 많음)
        - grayscale_similarity: RGB 채널 간 표준편차 평균 (낮을수록 그레이스케일)
        - edge_density: Canny 엣지 픽셀 비율 (0~1, 낮을수록 부드러운 그림)
        - mean_value: HSV 명도 평균 (0~255)
    """
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        return None

    # BGR → HSV 변환 (Hue: 색상, Saturation: 채도, Value: 명도)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    # BGR → 그레이스케일 변환 (엣지 검출용)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    h, s, v = cv2.split(img_hsv)  # HSV 각 채널 분리
    b, g, r = cv2.split(img_bgr)  # BGR 각 채널 분리

    # ---- 특징 1: 채도(Saturation) 평균 ----
    # 수묵화는 먹+물이라 채도가 매우 낮음 (보통 0~30 범위)
    mean_saturation = float(np.mean(s))

    # ---- 특징 2: 여백 비율 ----
    # 명도(V)가 200 이상인 픽셀 = 거의 흰색 = 여백
    # 수묵화는 여백의 미가 있어서 이 비율이 높음
    whitespace_ratio = float(np.mean(v > 200))

    # ---- 특징 3: 그레이스케일 유사도 ----
    # 각 픽셀에서 R, G, B 값의 표준편차를 구함
    # 수묵화는 R≈G≈B이므로 std가 매우 낮음
    # 채색화(단청, 민화 등)는 R,G,B 차이가 커서 std가 높음
    rgb_stack = np.stack([r, g, b], axis=-1).astype(np.float32)
    pixel_std = np.std(rgb_stack, axis=-1)
    grayscale_similarity = float(np.mean(pixel_std))

    # ---- 특징 4: 엣지 밀도 ----
    # Canny edge detection으로 경계선 검출
    # 수묵화는 번짐/농담 표현이 많아 엣지가 적음
    # 디지털 일러스트나 만화는 선이 뚜렷해서 엣지가 많음
    edges = cv2.Canny(img_gray, 50, 150)
    edge_density = float(np.mean(edges > 0))

    # ---- 특징 5: 평균 명도 ----
    # 참고용 (직접 필터링에는 안 쓰지만 분포 확인용)
    mean_value = float(np.mean(v))

    return {
        "path": str(img_path),
        "mean_saturation": round(mean_saturation, 2),
        "whitespace_ratio": round(whitespace_ratio, 4),
        "grayscale_similarity": round(grayscale_similarity, 2),
        "edge_density": round(edge_density, 4),
        "mean_value": round(mean_value, 2),
    }

In [ ]:
# 모든 이미지에 대해 수묵화 특징값을 추출하여 DataFrame으로 정리
if image_paths:
    features = []
    for p in tqdm(image_paths, desc="수묵화 특징 추출"):
        feat = compute_ink_features(p)
        if feat is not None:
            features.append(feat)

    df_feat = pd.DataFrame(features)
    print(f"특징 추출 완료: {len(df_feat)}장")
    # describe()로 각 특징값의 분포(평균, 표준편차, 최소/최대 등)를 한눈에 확인
    df_feat.describe()
else:
    print("이미지가 없으므로 건너뜁니다.")

In [ ]:
def plot_feature_distributions(df: pd.DataFrame):
    """
    5가지 특징값의 히스토그램을 그린다.
    빨간 점선 = 중앙값(median)
    → 이 분포를 보고 다음 셀의 THRESHOLDS 임계값을 조정하면 됨
    """
    feature_cols = ["mean_saturation", "whitespace_ratio", "grayscale_similarity", "edge_density", "mean_value"]
    titles = ["채도 평균", "여백 비율", "그레이스케일 유사도\n(낮을수록 무채색)", "엣지 밀도", "평균 명도"]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, col, title in zip(axes, feature_cols, titles):
        ax.hist(df[col], bins=50, edgecolor="black", alpha=0.7)
        ax.set_title(title)
        # 중앙값을 빨간 점선으로 표시
        ax.axvline(df[col].median(), color="red", linestyle="--", label=f"median={df[col].median():.2f}")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


if image_paths:
    plot_feature_distributions(df_feat)

In [ ]:
# ============================
# 수묵화 필터링 임계값 (Thresholds)
# ============================
# [중요] 위 히스토그램을 보고 아래 값을 데이터에 맞게 조정하세요!
# 너무 엄격하면 수묵화도 걸러지고, 너무 느슨하면 비수묵화가 포함됩니다.

THRESHOLDS = {
    # 채도 평균 상한: 이 값 이하 = 무채색(수묵화)
    # 수묵화는 보통 0~30, 채색화는 50 이상
    "max_saturation": 50,

    # 여백 비율 하한: 이 값 이상 = 여백이 충분함
    # 수묵화는 보통 0.2~0.7, 꽉 찬 그림은 0.05 이하
    "min_whitespace_ratio": 0.1,

    # 그레이스케일 유사도 상한: 이 값 이하 = R≈G≈B (무채색)
    # 수묵화는 보통 0~10, 채색화는 20 이상
    "max_grayscale_sim": 15,

    # 엣지 밀도 상한: 이 값 이하 = 부드러운 붓터치
    # 수묵화는 보통 0.02~0.10, 만화/일러스트는 0.15 이상
    "max_edge_density": 0.15,
}


def is_ink_painting(row: pd.Series, thresholds: dict = THRESHOLDS) -> bool:
    """
    4가지 조건을 모두 만족하면 수묵화로 판정 (AND 조건).
    하나라도 벗어나면 비수묵화로 분류됨.
    """
    return (
        row["mean_saturation"] <= thresholds["max_saturation"]
        and row["whitespace_ratio"] >= thresholds["min_whitespace_ratio"]
        and row["grayscale_similarity"] <= thresholds["max_grayscale_sim"]
        and row["edge_density"] <= thresholds["max_edge_density"]
    )


# 각 이미지에 수묵화 여부 판정 적용
if image_paths:
    df_feat["is_ink"] = df_feat.apply(is_ink_painting, axis=1)
    n_ink = df_feat["is_ink"].sum()
    n_total = len(df_feat)
    print(f"수묵화 판정: {n_ink} / {n_total} ({n_ink/n_total*100:.1f}%)")
    print(f"비수묵화 (제거 대상): {n_total - n_ink}장")

In [ ]:
def show_filtered_comparison(df: pd.DataFrame, n=8):
    """
    수묵화로 판정된 이미지(윗줄)와 제외된 이미지(아랫줄)를 비교 출력한다.
    → 필터링이 제대로 작동하는지 눈으로 확인하는 용도
    → 비수묵화 쪽에 수묵화가 섞여 있으면 THRESHOLDS 값을 조정해야 함
    """
    ink_paths = df[df["is_ink"]]["path"].tolist()
    non_ink_paths = df[~df["is_ink"]]["path"].tolist()

    fig, axes = plt.subplots(2, n, figsize=(n * 2.5, 6))

    # 윗줄: 수묵화 판정 (O 표시)
    for i in range(n):
        if i < len(ink_paths):
            img = Image.open(ink_paths[i])
            axes[0][i].imshow(img)
            axes[0][i].set_title("O", fontsize=10, color="green")
        axes[0][i].axis("off")
    axes[0][0].set_ylabel("수묵화", fontsize=12)

    # 아랫줄: 비수묵화 판정 (X 표시)
    for i in range(n):
        if i < len(non_ink_paths):
            img = Image.open(non_ink_paths[i])
            axes[1][i].imshow(img)
            axes[1][i].set_title("X", fontsize=10, color="red")
        axes[1][i].axis("off")
    axes[1][0].set_ylabel("비수묵화", fontsize=12)

    plt.suptitle("수묵화 필터링 결과 비교", fontsize=14)
    plt.tight_layout()
    plt.show()


if image_paths:
    show_filtered_comparison(df_feat)

## 3. 전처리 적용

In [ ]:
# ============================
# 전처리 설정
# ============================
TARGET_SIZE = 512   # Stable Diffusion 입력 크기 기준 (512x512)
MIN_RESOLUTION = 256  # 가로 또는 세로가 256px 미만이면 품질 부족으로 제외


def preprocess_image(img_path: str, target_size: int = TARGET_SIZE) -> np.ndarray | None:
    """
    이미지 한 장을 전처리한다.

    처리 과정:
    1. 저해상도(256px 미만) 이미지는 제외 (None 반환)
    2. 긴 변 기준으로 target_size에 맞게 비율 유지 리사이즈
    3. 남는 영역은 흰색(255)으로 패딩 → 정사각형 출력
       (수묵화 배경이 흰색이므로 패딩이 자연스러움)

    Args:
        img_path: 이미지 파일 경로
        target_size: 출력 이미지 크기 (가로=세로)

    Returns:
        전처리된 numpy 배열 (target_size x target_size x 3) 또는 None
    """
    img = cv2.imread(img_path)
    if img is None:
        return None

    h, w = img.shape[:2]

    # 너무 작은 이미지는 학습에 부적합하므로 제외
    if min(h, w) < MIN_RESOLUTION:
        return None

    # 비율 유지 리사이즈: 긴 변을 target_size에 맞춤
    scale = target_size / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # 흰색(255) 캔버스에 중앙 배치 → 정사각형 만들기
    canvas = np.full((target_size, target_size, 3), 255, dtype=np.uint8)
    y_offset = (target_size - new_h) // 2  # 상하 여백
    x_offset = (target_size - new_w) // 2  # 좌우 여백
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized

    return canvas

In [ ]:
# 수묵화로 판정된 이미지만 전처리하여 data/processed/에 저장
if image_paths:
    ink_df = df_feat[df_feat["is_ink"]].copy()
    ink_image_paths = ink_df["path"].tolist()

    processed_count = 0
    skipped_count = 0

    for img_path in tqdm(ink_image_paths, desc="전처리 중"):
        result = preprocess_image(img_path)
        if result is not None:
            # 원본 파일명 유지, 확장자만 png로 통일
            filename = Path(img_path).stem + ".png"
            save_path = PROCESSED_DIR / filename
            cv2.imwrite(str(save_path), result)
            processed_count += 1
        else:
            skipped_count += 1

    print(f"\n전처리 완료: {processed_count}장 저장")
    print(f"저해상도 제외: {skipped_count}장")
    print(f"저장 경로: {PROCESSED_DIR.resolve()}")
else:
    print("이미지가 없으므로 건너뜁니다.")

## 4. ControlNet용 스케치 추출

Canny Edge Detection으로 스케치 라인을 추출합니다.  
나중에 HED 등 더 정교한 방법으로 교체할 수 있습니다.

In [ ]:
def extract_sketch(img_path: str, low_thresh: int = 30, high_thresh: int = 100) -> np.ndarray | None:
    """
    Canny Edge Detection으로 스케치(라인 드로잉)를 추출한다.

    ControlNet의 Canny 모델에 입력할 스케치 이미지를 생성하는 함수.
    수묵화의 윤곽선만 추출하여 흰 배경 + 검은 라인 형태로 출력.

    처리 과정:
    1. 그레이스케일로 읽기
    2. 가우시안 블러로 노이즈 제거 (불필요한 잡선 방지)
    3. Canny edge detection으로 윤곽선 검출
    4. 반전 (Canny 출력: 검은 배경+흰 라인 → 흰 배경+검은 라인)

    Args:
        img_path: 이미지 파일 경로
        low_thresh: Canny 하한 임계값 (낮을수록 더 많은 엣지 검출)
        high_thresh: Canny 상한 임계값 (높을수록 강한 엣지만 검출)
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 가우시안 블러: 5x5 커널로 노이즈 제거
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    # Canny edge detection
    edges = cv2.Canny(blurred, low_thresh, high_thresh)

    # 반전: 흰 배경 + 검은 라인 (ControlNet Canny 입력 형식)
    sketch = 255 - edges
    return sketch

In [ ]:
# 전처리 완료된 이미지(data/processed/)에서 스케치를 추출하여 data/sketches/에 저장
# 파일명은 원본과 동일하게 유지 (1:1 매칭)
if image_paths:
    processed_paths = sorted(PROCESSED_DIR.glob("*.png"))

    sketch_count = 0
    for p in tqdm(processed_paths, desc="스케치 추출 중"):
        sketch = extract_sketch(str(p))
        if sketch is not None:
            save_path = SKETCH_DIR / p.name  # 동일 파일명으로 저장
            cv2.imwrite(str(save_path), sketch)
            sketch_count += 1

    print(f"스케치 추출 완료: {sketch_count}장")
    print(f"저장 경로: {SKETCH_DIR.resolve()}")
else:
    print("이미지가 없으므로 건너뜁니다.")

In [ ]:
def show_original_vs_sketch(processed_dir: Path, sketch_dir: Path, n=4):
    """
    전처리된 원본(윗줄)과 추출된 스케치(아랫줄)를 나란히 비교한다.
    → 스케치 품질이 괜찮은지 눈으로 확인하는 용도
    → 라인이 너무 적거나 많으면 extract_sketch()의 low_thresh, high_thresh 조정
    """
    processed_paths = sorted(processed_dir.glob("*.png"))[:n]

    fig, axes = plt.subplots(2, n, figsize=(n * 3, 6))
    for i, p in enumerate(processed_paths):
        # 윗줄: 전처리된 원본 이미지
        img = cv2.imread(str(p))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # OpenCV는 BGR이라 RGB로 변환
        axes[0][i].imshow(img_rgb)
        axes[0][i].set_title(p.name, fontsize=8)
        axes[0][i].axis("off")

        # 아랫줄: 스케치 이미지
        sketch_path = sketch_dir / p.name
        if sketch_path.exists():
            sketch = cv2.imread(str(sketch_path), cv2.IMREAD_GRAYSCALE)
            axes[1][i].imshow(sketch, cmap="gray")
        axes[1][i].axis("off")

    axes[0][0].set_ylabel("원본", fontsize=12)
    axes[1][0].set_ylabel("스케치", fontsize=12)
    plt.suptitle("원본 vs 스케치 비교", fontsize=14)
    plt.tight_layout()
    plt.show()


if image_paths:
    show_original_vs_sketch(PROCESSED_DIR, SKETCH_DIR)

## 5. 메타데이터 & 통계 리포트

In [ ]:
# ============================
# 최종 결과 요약 & CSV 리포트 저장
# ============================
if image_paths:
    print("=" * 50)
    print("        전처리 결과 요약")
    print("=" * 50)
    print(f"원본 이미지 수:        {len(image_paths)}")
    print(f"수묵화 판정:           {df_feat['is_ink'].sum()}")
    print(f"비수묵화 제외:         {(~df_feat['is_ink']).sum()}")
    print(f"전처리 저장:           {len(list(PROCESSED_DIR.glob('*.png')))}")
    print(f"스케치 추출:           {len(list(SKETCH_DIR.glob('*.png')))}")
    print("=" * 50)

    # 각 이미지의 특징값 + 수묵화 판정 결과를 CSV로 저장
    # → 팀원과 공유하거나, 나중에 임계값 재조정할 때 활용
    report_path = PROJECT_ROOT / "outputs" / "preprocessing_report.csv"
    df_feat.to_csv(report_path, index=False)
    print(f"\n특징값 리포트 저장: {report_path.resolve()}")
else:
    print("데이터가 없습니다. data/raw/ 에 이미지를 넣은 후 처음부터 다시 실행하세요.")